03_results_comparison.ipynb (porovnanie výsledkov)
Cieľ notebooku
- načítať outputs/summary_metrics.json,
- spraviť tabuľku výsledkov,
- spraviť barploty pre ACC/F1/AUC,
- ak máte aj ROC dáta uložené, doplniť ROC (ak nie, tak aspoň metriky).

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath("..")  # projekt/
SUMMARY_PATH = os.path.join(PROJECT_ROOT, "outputs", "summary_metrics.json")

with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = json.load(f)

summary["generated_at"], list(summary["results"].keys())

In [ ]:
results = summary["results"]  # { "tess_mlp": {...}, ... }

rows = []
for tag, m in results.items():
    # tag = "tess_mlp", "ws3d_cnn", ...
    dataset, model = tag.split("_", 1)
    row = {"tag": tag, "dataset": dataset, "model": model}
    row.update(m)
    rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
# --- Vyber stĺpcov (podľa toho, čo reálne máte v metrics dict) ---
cols = ["dataset", "model", "accuracy", "f1", "precision", "recall", "roc_auc"]
available = [c for c in cols if c in df.columns]
df_view = df[available].sort_values(["dataset", "model"])
df_view

In [ ]:
# --- Barploty ---
def barplot(metric):
    if metric not in df.columns:
        print(f"Metric '{metric}' not in summary.")
        return
    pivot = df.pivot(index="dataset", columns="model", values=metric)
    pivot.plot(kind="bar", figsize=(7,4))
    plt.title(metric)
    plt.ylabel(metric)
    plt.grid(axis="y", alpha=0.3)
    plt.show()

for metric in ["accuracy", "f1", "roc_auc"]:
    barplot(metric)

In [ ]:
# --- Export tabuľky do CSV pre report ---
OUT_CSV = os.path.join(PROJECT_ROOT, "outputs", "results_table.csv")
df_view.to_csv(OUT_CSV, index=False)
OUT_CSV